# Extractor Agent

This notebook extracts atomic clinical facts from original MTSamples notes. It does not simplify or interpret the notes. Each atom includes the exact source text and a category for downstream agents.

## Output format

```json
{
  "atoms": [
    {
      "text": "denies chest pain",
      "type": "negative_finding",
      "source_quote": "denies chest pain"
    }
  ],
  "not_found_types": ["diagnosis"]
}
```

In [20]:
import json
import os
import re
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    types = None

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "mtsamples_baseline_outputs.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "extractor_outputs.jsonl"
MODEL = "gemini-3.5-flash"

ALLOWED_TYPES = {
    "diagnosis", "diagnosis_reasoning", "symptom", "risk", "medical_history",
    "family_history", "environmental_history", "demographic", "medical_specialty",
    "examination", "examination_finding", "lab_value", "vital_sign", "procedure",
    "treatment", "medication", "referral", "follow_up", "patient_education",
}
DATA_PATH

WindowsPath('C:/Users/aliam/OneDrive/Desktop/Google 2A/Google-2A-faithfulmed/data/mtsamples_baseline_outputs.csv')

In [21]:
EXTRACTOR_INSTRUCTIONS = """
You extract atomic clinical facts from a clinical note.

Input: JSON containing clinical_note.

Output: Return only valid JSON with exactly these fields:
{
  "atoms": [
    {"text": "...", "type": "...", "source_quote": "..."}
  ],
  "not_found_types": []
}

Extraction rules
-   Use only the source.
    Extract explicitly stated clinical facts without inferring, diagnosing, explaining, or adding information.
    Do not use outside knowledge.
-   Each atom must represent only one finding, symptom, diagnosis, medication, lab value, or other distinct fact.
    Split lists into separate atoms when each item is a distinct fact.
    Preserve qualifiers and relationships necessary to retain the original meaning after splitting.
-   source_quote: an exact contiguous quote from the clinical note that provides enough context to support the complete atom.
    Do not use an isolated phrase as the source_quote when the surrounding text is needed to establish its meaning or create a complete sentence.
-   Text should be a concise statement using only minimal grammatical normalization.
    Do not expand abbreviations, correct typos, or change wording.
-   Preserve status and details.
    Retain all negation, uncertainty, temporality, status, numbers, units, doses, routes, frequencies, and other qualifiers.
    Never assume unstated details.
-   Refer to the individual as "patient" in the extracted text.
    Do not extract names, dates of birth, addresses, phone numbers, or other identifiers.
-   Assign one allowed category per atom, choosing the most appropriate category:
    - diagnosis
    - diagnosis_reasoning
    - symptom
    - risk
    - medical_history
    - family_history
    - environmental_history
    - demographic
    - medical_specialty
    - examination
    - examination_finding
    - lab_value
    - vital_sign
    - procedure
    - treatment
    - medication
    - referral
    - follow_up
    - patient_education
-   Report missing categories.
    not_found_types must list every allowed category that has no supporting text in the note.
    Do not create atoms for missing or unknown information.
-   Return JSON only.
    If no facts are found, return an empty atoms array and list all categories in not_found_types.
-   The source_quote must be copied exactly from the note.
-   The text may use minimal grammatical normalization for clarity.
-   Validate traceability using source_quote, not text.
-   Preserve family relationships exactly. "Father's side" is not the same as "father."
-   Preserve negation.
-   Preserve if a medication is indicated to be existing, stopped, new
-   Use negative_finding for explicit denials and absent findings.

Example
Source: "Significant for high blood pressure and diabetes as well as cancer on her father side"
Extract separate atoms for high blood pressure, diabetes, and cancer on the patient's father's side. Preserve the family relationship in the cancer atom.

The clinical note is provided in the input JSON as clinical_note.
"""

# EXTRACTOR_INSTRUCTIONS = """
# You extract atomic clinical facts from a clinical note.

# Input: JSON containing clinical_note.

# Output: Return only valid JSON with exactly these fields:
# {
#   "atoms": [
#     {"text": "...", "type": "...", "source_quote": "..."}
#   ],
#   "not_found_types": []
# }

# Extraction rules
# -   Use only the source.
#     Extract explicitly stated clinical facts without inferring, diagnosing, explaining, or adding information.
#     Do not use outside knowledge.
# -   Keep facts atomic.
#     Each atom must represent only one finding, symptom, diagnosis, medication, lab value, or other distinct fact.
#     Split lists into separate atoms when each item is a distinct fact.
#     Preserve qualifiers and relationships necessary to retain the original meaning after splitting.
# -   source_quote: an exact contiguous quote from the clinical note that provides enough context to support the complete atom, without being too long.
#     Do not use an isolated term as the source_quote when the surrounding text is needed to establish its meaning.
# -   Text should be a concise statement using only minimal grammatical normalization.
#     Do not expand abbreviations, correct typos, or change wording in ways that alter meaning.
# -   Preserve status and details.
#     Retain all negation, uncertainty, temporality, status, numbers, units, doses, routes, frequencies, and other qualifiers.
#     Distinguish past, stopped, resolved, suspected, negated, planned, and recommended information from current or confirmed facts.
#     Never assume unstated details.
# -   Refer to the individual as "patient" in the extracted text.
#     Do not extract names, dates of birth, addresses, phone numbers, or other identifiers.
# -   Assign one allowed category per atom, choosing the most appropriate category:
#     - diagnosis
#     - diagnosis_reasoning
#     - symptom
#     - risk
#     - medical_history
#     - family_history
#     - environmental_history
#     - demographic
#     - medical_specialty
#     - examination
#     - examination_finding
#     - lab_value
#     - vital_sign
#     - procedure
#     - treatment
#     - medication
#     - referral
#     - follow_up
#     - patient_education
# -   Report missing categories.
#     not_found_types must list every allowed category that has no supporting text in the note.
#     Do not create atoms for missing or unknown information.
# -   Return JSON only.
#     If no facts are found, return an empty atoms array and list all categories in not_found_types.
# -   The source_quote must be copied exactly from the note.
# -   The text may use minimal grammatical normalization for clarity.
# -   Validate traceability using source_quote, not text.
# -   Preserve family relationships exactly. "Father's side" is not the same as "father."
# -   Preserve negation.
# -   Preserve if a medication is indicated to be existing, stopped, new
# -   Use negative_finding for explicit denials and absent findings.

# Example
# Source: "Significant for high blood pressure and diabetes as well as cancer on her father side"
# Extract separate atoms for high blood pressure, diabetes, and cancer on the patient's father's side. Preserve the family relationship in the cancer atom.

# The clinical note is provided in the input JSON as clinical_note.
# """
# EXTRACTOR_INSTRUCTIONS = """
# You extract atomic clinical facts from a clinical note.

# Input: JSON containing clinical_note.

# Output: Return only valid JSON with exactly these fields:
# {
#   "atoms": [
#     {"text": "...", "type": "...", "source_quote": "..."}
#   ],
#   "not_found_types": []
# }

# Extraction rules
# -   Use only the source.
#     Extract explicitly stated clinical facts without inferring, diagnosing, explaining, or adding information.
#     Do not use outside knowledge.
# -   Keep facts atomic.
#     Each atom must represent one finding, symptom, diagnosis, medication, lab value, or other distinct fact.
#     Split lists into separate atoms when each item is a distinct fact.
#     Preserve qualifiers and relationships necessary to retain the original meaning after splitting.
# -   source_quote must contain the exact original words supporting the atom.
#     Text should be a concise statement using only minimal grammatical normalization.
#     Do not expand abbreviations, correct typos, or change wording in ways that alter meaning.
# -   Preserve status and details.
#     Retain all negation, uncertainty, temporality, status, numbers, units, doses, routes, frequencies, and other qualifiers.
#     Distinguish past, stopped, resolved, suspected, negated, planned, and recommended information from current or confirmed facts.
#     Never assume unstated details.
# -   Refer to the individual as "patient" in the extracted text.
#     Do not extract names, dates of birth, addresses, phone numbers, or other identifiers.
# -   Assign one allowed category per atom, choosing the most appropriate category:
#     - diagnosis
#     - diagnosis_reasoning
#     - symptom
#     - risk
#     - medical_history
#     - family_history
#     - environmental_history
#     - demographic
#     - medical_specialty
#     - examination
#     - examination_finding
#     - lab_value
#     - vital_sign
#     - procedure
#     - treatment
#     - medication
#     - referral
#     - follow_up
#     - patient_education
# -   Report missing categories.
#     not_found_types must list every allowed category that has no supporting text in the note.
#     Do not create atoms for missing or unknown information.
# -   Return JSON only.
#     If no facts are found, return an empty atoms array and list all categories in not_found_types.
# -   The source_quote must be copied exactly from the note.
# -   The text may use minimal grammatical normalization for clarity.
# -   Validate traceability using source_quote, not text.
# -   Preserve family relationships exactly. "Father's side" is not the same as "father."
# -   Preserve negation.
# -   Preserve if a medication is indicated to be existing, stopped, new
# -   Use negative_finding for explicit denials and absent findings.

# Example
# Source: "Significant for high blood pressure and diabetes as well as cancer on her father side"
# Extract separate atoms for high blood pressure, diabetes, and cancer on the patient's father's side. Preserve the family relationship in the cancer atom.

# The clinical note is provided in the input JSON as clinical_note.
# """

RESPONSE_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "atoms": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "text": {"type": "STRING"},
                    "type": {"type": "STRING", "enum": sorted(ALLOWED_TYPES)},
                    "source_quote": {"type": "STRING"},
                },
                "required": ["text", "type", "source_quote"],
            },
        },
        "not_found_types": {
            "type": "ARRAY",
            "items": {"type": "STRING", "enum": sorted(ALLOWED_TYPES)},
        },
    },
    "required": ["atoms", "not_found_types"],
}


In [22]:
def normalize_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


def quote_is_in_source(source_text: str, source_quote: str) -> bool:
    if not isinstance(source_text, str) or not isinstance(source_quote, str):
        return False
    if not source_quote.strip():
        return False
    return source_quote in source_text or normalize_whitespace(source_quote) in normalize_whitespace(source_text)


def validate_output(model_output: dict[str, Any], source_text: str) -> dict[str, Any]:
    if not isinstance(model_output, dict):
        print("Extractor response must be a JSON object.")
    atoms = model_output.get("atoms")
    not_found_types = model_output.get("not_found_types")
    if not isinstance(atoms, list) or not isinstance(not_found_types, list):
        print("Extractor response must contain atoms and not_found_types lists.")
    invalid_not_found = set(not_found_types) - ALLOWED_TYPES
    if invalid_not_found:
        print(f"Invalid not_found_types: {sorted(invalid_not_found)}")
    validated_atoms = []
    for index, atom in enumerate(atoms):
        if not isinstance(atom, dict):
            print(f"Atom {index} is not an object.")
        required = {"text", "type", "source_quote"}
        missing = required - atom.keys()
        if missing:
            print(f"Atom {index} is missing fields: {sorted(missing)}")
        if atom["type"] not in ALLOWED_TYPES:
            print(f"Atom {index} has invalid type: {atom['type']}")
        if not all(isinstance(atom[field], str) and atom[field].strip() for field in required):
            print(f"Atom {index} has an empty or non-string field.")
        if not quote_is_in_source(source_text, atom["source_quote"]):
            print(f"Atom {index} source_quote was not found in the source text.")
        validated_atoms.append({field: atom[field].strip() for field in required})
    present_types = {atom["type"] for atom in validated_atoms}
    expected_not_found = ALLOWED_TYPES - present_types
    if set(not_found_types) != expected_not_found:
        print("not_found_types must contain exactly the categories absent from atoms.")
    return {"atoms": validated_atoms, "not_found_types": sorted(not_found_types)}


In [23]:
records = pd.read_csv(DATA_PATH)
required_columns = {"sample_name", "transcription"}
missing_columns = required_columns - set(records.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
records = records[["sample_name", "transcription"]].dropna()
records = records[records["transcription"].str.strip().ne("")].reset_index(drop=True)
print(f"Loaded {len(records):,} original MTSamples notes.")
print(records[["sample_name"]].head())

Loaded 50 original MTSamples notes.
                                      sample_name
0   Bariatric Consult - Surgical Weight Loss - 4 
1                  Consult - Atrial Fibrillation 
2              Consult - Atrial Fibrillation - 1 
3                     Allergy Evaluation Consult 
4                       Cardiac Consultation - 2 


In [24]:
def make_client():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("Set GEMINI_API_KEY before running Gemini extraction.")
    if genai is None or types is None:
        raise ImportError("Install the existing google-genai dependency first.")
    return genai.Client(api_key=api_key)


def extract_atoms(source_text: str, client: Any | None = None) -> dict[str, Any]:
    if client is None:
        client = make_client()
    prompt = EXTRACTOR_INSTRUCTIONS + "\n\nINPUT JSON:\n" + json.dumps({"clinical_note": source_text})
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RESPONSE_SCHEMA,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        ),
    )
    if not response.text:
        raise ValueError("Gemini returned an empty response.")
    model_output = json.loads(response.text)
    return validate_output(model_output, source_text)


In [25]:
RUN_REAL_EXTRACTION = True
sample_source = records.iloc[0]["transcription"]
if RUN_REAL_EXTRACTION:
    sample_output = extract_atoms(sample_source)
    print(json.dumps(sample_output, indent=2))

{
  "atoms": [
    {
      "type": "demographic",
      "source_quote": "Ms. A is a 55-year-old female who presented to the Bariatric Surgery Service for consideration of laparoscopic Roux-en-Y gastric bypass.",
      "text": "Patient is a 55-year-old female."
    },
    {
      "type": "medical_specialty",
      "source_quote": "Ms. A is a 55-year-old female who presented to the Bariatric Surgery Service for consideration of laparoscopic Roux-en-Y gastric bypass.",
      "text": "Patient presented to the Bariatric Surgery Service."
    },
    {
      "type": "procedure",
      "source_quote": "Ms. A is a 55-year-old female who presented to the Bariatric Surgery Service for consideration of laparoscopic Roux-en-Y gastric bypass.",
      "text": "Patient presented for consideration of laparoscopic Roux-en-Y gastric bypass."
    },
    {
      "type": "medical_history",
      "source_quote": "The patient states that she has been overweight for approximately 35 years and has tried multipl

In [26]:
def run_extraction(records_to_process: pd.DataFrame, output_path: Path, limit: int = 5) -> None:
    client = make_client()
    selected = records_to_process.head(limit)
    with output_path.open("w", encoding="utf-8") as output_file:
        for row in selected.itertuples(index=False):
            extraction = extract_atoms(row.transcription, client=client)
            output_file.write(json.dumps({
                "sample_name": row.sample_name,
                **extraction,
            }) + "\n")
    print(f"Saved {len(selected)} extraction results to {output_path}")


# Uncomment after testing one real note.
# run_extraction(records, OUTPUT_PATH, limit=5)